# start_here: pySTAMPS repo guide

This notebook is the repo-level entrypoint for `pySTAMPS`. It explains what the project does, which tools exist, how the repo is organized, where the Rust/native backend fits, and which path to follow depending on whether you are a user, maintainer, or backend developer.

Use this notebook when you want one compact map of the project before opening the more specialized notebooks. It is the notebook the rest of the set should be read from.

## What this repo is for

`pySTAMPS` is a Python-first runtime for StaMPS-style InSAR processing, dataset inspection, verification, and parity work during migration away from legacy workflows.

Three common applications:
- Run a dataset through stages 1 to 8 on a safe local copy.
- Compare a run against trusted reference outputs with explicit tolerances.
- Experiment with kernel backends, especially the stage-2 Rust/native path, without losing parity discipline.

Primary interfaces:
- CLI for most users.
- Notebooks for guided learning and audit narratives.
- A small internal Python API for orchestration, verification, and kernel inspection.

In [ ]:
from pathlib import Path

REPO_ROOT = Path.cwd().resolve()
NOTEBOOKS = sorted((REPO_ROOT / 'notebooks').glob('*.ipynb'))
DATASETS = [
    REPO_ROOT / 'inputs_and_outputs' / 'InSAR_dataset_test',
    REPO_ROOT / 'inputs_and_outputs' / 'InSAR_dataset_test_stage8diag',
    REPO_ROOT / 'inputs_and_outputs' / 'InSAR_dataset_test_stage8diag_hl',
]

print('repo_root:', REPO_ROOT)
print('\nnotebooks:')
for path in NOTEBOOKS:
    print(' -', path.name)

print('\ndatasets:')
for path in DATASETS:
    print(f' - {path.name}:', 'present' if path.exists() else 'missing')


## Visual maps

Two Figma diagrams complement this notebook:

- [pySTAMPS Data Flow](https://www.figma.com/online-whiteboard/create-diagram/bffb9822-cd33-4ff6-a843-8157bf91607f?utm_source=chatgpt&utm_content=edit_in_figjam&oai_id=&request_id=a0724491-4854-4a20-ac5c-7bf47859e4d6)
- [pySTAMPS Backend Resolution](https://www.figma.com/online-whiteboard/create-diagram/51f59245-9574-4e07-9918-32089ed34a82?utm_source=chatgpt&utm_content=edit_in_figjam&oai_id=&request_id=dac3a4cd-1378-4f08-97e8-3e865c497368)

Read them this way:
- The data-flow diagram shows how a dataset moves from inspection, through patch and merged stages, into verification and audit evidence.
- The backend-resolution diagram shows how config and kernel registry choices determine whether work executes on Python, native, or CUDA-backed paths.

## Mental model

A `pySTAMPS` run is centered on one StaMPS-style dataset root.

The practical model is:
- `PATCH_*` directories hold patch-scoped artifacts.
- Stages 1 to 5 are patch-heavy.
- Stages 6 to 8 are merged or dataset-level products.
- The normal user loop is `status` -> `run` -> `verify`.
- The stricter maintainer loop is `status` -> `run/audit` -> `verify` -> artifact review.

If you only remember one rule: run on a copy of your dataset, not your only original.

## Repo map

| Area | Why it exists |
| --- | --- |
| `pystamps/cli.py` | CLI entrypoint for `status`, `run`, `verify`, `describe-inputs`, `describe-backends`, and `list-legacy`. |
| `pystamps/pipeline/` | Stage orchestration plus the ported Python implementations. |
| `pystamps/kernels/` | Backend registry and accelerated kernels, including the Rust/native extension surface. |
| `pystamps/verify.py` | Numeric comparison against golden datasets under explicit tolerances. |
| `pystamps/io/` | Dataset discovery and MAT-file access helpers. |
| `scripts/` | Audit, benchmark, and parity-maintenance utilities. |
| `notebooks/` | Guided notebooks for learning, stage walkthroughs, verification, and native parity. |
| `tests/` | Contract coverage for CLI, kernels, validation, and dataset expectations. |
| `inputs_and_outputs/` | Optional local sample and parity datasets used by notebooks and audits. |

## Stage map

The shipped pipeline stages are:

| Stage | Name | Scope |
| --- | --- | --- |
| 1 | Initial load | Patch |
| 2 | Estimate gamma | Patch |
| 3 | Select PS pixels | Patch |
| 4 | Weed adjacent pixels | Patch |
| 5 | Correct phase + merge | Patch, then merged products |
| 6 | Unwrap phase | Merged |
| 7 | Calculate SCLA | Merged |
| 8 | Filter SCN | Merged |

Interpretation:
- Early stages prepare and select useful pixels.
- Middle stages consolidate patch results into dataset-level products.
- Late stages produce corrected, unwrapped, and filtered outputs for downstream analysis.

## Main CLI tools

The CLI is the recommended interface for most users:

- `pystamps status`: inspect inferred patch and merged stage state.
- `pystamps run`: dry-run or execute a stage range on a dataset.
- `pystamps verify`: compare a run folder against a golden folder.
- `pystamps describe-inputs`: inspect the logical inputs required by one stage or all stages.
- `pystamps describe-backends`: inspect backend providers and per-kernel coverage in the current environment.
- `pystamps list-legacy`: discover legacy scripts in a StaMPS checkout.

In [ ]:
CLI_COMMANDS = [
    ('status', 'Inspect stage progress in a dataset'),
    ('run', 'Execute or dry-run a stage range'),
    ('verify', 'Compare a run folder against a golden folder'),
    ('describe-inputs', 'Explain required inputs for one stage or all stages'),
    ('describe-backends', 'Show registered backend providers and kernel coverage'),
    ('list-legacy', 'List discoverable legacy StaMPS scripts'),
]

for name, description in CLI_COMMANDS:
    print(f'{name:18} {description}')


## Configuration and backend choices

The most important runtime knobs are in `RunConfig.runtime`:

- `backend`: high-level execution mode such as `auto`, `threads`, `processes`, `gpu`, or `native`.
- `stage2_kernel_backend`: force stage 2 to `python` or `native`.
- `stage2_patch_backend_overrides`: choose stage-2 backend per patch.
- `kernel_backend_overrides`: pin specific kernels such as stage 7 or stage 8.
- `stage2_native_threads`: `0` means use the full detected CPU budget for stage 2 by default.
- `stage7_chunk_ps` and `stage8_chunk_edges`: tune chunk sizes for later kernels.

Tolerance and compatibility live elsewhere:
- `tolerance`: numeric comparison settings for verification.
- `compat.reference_root` and `compat.strict_reference`: replay or parity-oriented compatibility modes.

In [ ]:
import json
from pystamps.kernels import describe_backend_matrix

matrix = describe_backend_matrix()
print(json.dumps(matrix, indent=2)[:5000])


## Rust and native backend story

The native story is parity-first and performance-measured.

What that means in practice:
- Source installs build the stage-2 native extension with Rust.
- PyPI wheels may already include the extension for supported platforms.
- Native paths are exposed only where the repo considers them safe enough for the active contract.
- When a native path is unavailable or not yet exact enough, the runtime falls back to the Python reference implementation.
- Stage 4, 7, and 8 also have native exports where supported by the compiled extension.
- Notebook `04_rust_end2end_parity_validation.ipynb` shows the Rust exports, focused MATLAB StaMPS parity evidence, and Python-vs-Rust speed tables for the native stage 4, 7, and 8 kernels.

If you want the truth for your local checkout, trust `pystamps describe-backends` and the notebook 04 benchmark output more than assumptions.


## Typical workflows

### New user
1. Copy a dataset.
2. Run `status`.
3. Dry-run a stage range.
4. Execute the desired stages.

### Validation user
1. Produce a run folder.
2. Run `verify` against a trusted golden dataset.
3. Inspect failed paths if any.

### Maintainer
1. Run fresh-clone checks.
2. Run implementation tests.
3. Run `audit` or `verify` on maintained datasets when available.
4. Use notebooks to document or explain evidence.

### Backend developer
1. Compare Python and native results.
2. Inspect backend coverage.
3. Use parity notebooks and kernel tests before widening an accelerated path.

In [ ]:
MAKE_TARGETS = [
    ('make setup', 'sync the uv environment'),
    ('make test', 'run the main pytest suite'),
    ('make test-impl', 'run the implementation-focused test subset'),
    ('make build', 'build sdist and wheel artifacts'),
    ('make twine-check', 'validate built distributions'),
    ('make audit', 'run the strict parity audit on maintained datasets'),
    ('make verify', 'run the maintained reference verify check'),
    ('make benchmark', 'benchmark selected backends on the maintained stage-8 dataset'),
]

for command, description in MAKE_TARGETS:
    print(f'{command:18} {description}')


## Full parity against the maintained test sample

The repo's full maintained sample is `inputs_and_outputs/InSAR_dataset_test`.

This section shows the canonical parity view for that sample:
- read the latest full-audit artifact if it exists,
- extract the full-sample run root and derived stage timings,
- rerun explicit `pystamps verify` against the maintained full sample.

This notebook does not regenerate the expensive audit by default. It consumes the maintained artifact under `inputs_and_outputs/validation_runs/latest_audit.json` when present. If you need a fresh parity regeneration, move to notebook `03`.


In [ ]:
import json
import os
import subprocess
import sys
import time

import pandas as pd
from IPython.display import display

TIMING_CMD = [
    sys.executable,
    'scripts/derive_audit_stage_timings.py',
    '--audit',
    'inputs_and_outputs/validation_runs/latest_audit.json',
]

AUDIT_JSON = REPO_ROOT / 'inputs_and_outputs' / 'validation_runs' / 'latest_audit.json'
FULL_DATASET = REPO_ROOT / 'inputs_and_outputs' / 'InSAR_dataset_test'
FALLBACK_RUN_ROOT = REPO_ROOT / 'inputs_and_outputs' / 'RUN_FULL_GATE_1e10'
PARITY_ENV = os.environ.copy()
PARITY_ENV.update({
    'OPENBLAS_NUM_THREADS': '1',
    'OMP_NUM_THREADS': '1',
    'MKL_NUM_THREADS': '1',
    'PYTHONPATH': str(REPO_ROOT),
})

audit_payload = json.loads(AUDIT_JSON.read_text(encoding='utf-8')) if AUDIT_JSON.exists() else None
full_audit = None if audit_payload is None else next((audit for audit in audit_payload.get('audits', []) if audit.get('dataset') == 'InSAR_dataset_test'), None)
run_root = Path(full_audit['run_root']) if full_audit is not None else FALLBACK_RUN_ROOT if FALLBACK_RUN_ROOT.exists() else None

parity_rows = [
    {
        'audit_json_present': AUDIT_JSON.exists(),
        'audit_generated_at_utc': None if audit_payload is None else audit_payload.get('generated_at_utc'),
        'audit_ok': None if audit_payload is None else audit_payload.get('ok'),
        'full_sample_status': None if full_audit is None else full_audit.get('status'),
        'full_sample_checked': None if full_audit is None else full_audit.get('checked'),
        'full_sample_failed': None if full_audit is None else full_audit.get('failed'),
        'run_root': None if run_root is None else str(run_root),
        'golden_root': str(FULL_DATASET),
        'run_source': None if full_audit is None else full_audit.get('run_source'),
    }
]
display(pd.DataFrame(parity_rows))

timing_df = pd.DataFrame()
if AUDIT_JSON.exists() and full_audit is not None:
    timing_proc = subprocess.run(TIMING_CMD, cwd=REPO_ROOT, env=PARITY_ENV, capture_output=True, text=True)
    if timing_proc.returncode != 0:
        raise RuntimeError(timing_proc.stderr[-4000:])
    timing_payload = json.loads(timing_proc.stdout)
    full_timing = next((audit for audit in timing_payload.get('audits', []) if audit.get('dataset') == 'InSAR_dataset_test'), None)
    if full_timing is not None:
        timing_df = pd.DataFrame(full_timing['timings'])
        display(timing_df[['stage', 'duration_sec', 'artifact_count', 'basis']])

if run_root is None:
    print('No full-sample run root was available; parity verify skipped.')
else:
    verify_cmd = [
        'uv', 'run', 'pystamps', 'verify',
        '--run', str(run_root),
        '--golden', str(FULL_DATASET),
    ]
    started = time.perf_counter()
    verify_proc = subprocess.run(verify_cmd, cwd=REPO_ROOT, env=PARITY_ENV, capture_output=True, text=True)
    verify_elapsed_sec = time.perf_counter() - started
    if verify_proc.returncode not in (0, 1):
        raise RuntimeError(verify_proc.stderr[-4000:])
    verify_payload = json.loads(verify_proc.stdout)
    verify_summary = pd.DataFrame([
        {
            'verify_ok': verify_payload.get('ok'),
            'checked': verify_payload.get('checked'),
            'failed_count': len(verify_payload.get('failed', [])),
            'verify_returncode': verify_proc.returncode,
            'verify_elapsed_sec': round(verify_elapsed_sec, 2),
        }
    ])
    display(verify_summary)
    display(pd.DataFrame(verify_payload.get('failed', []))[:5])

    print('verify command:')
    print(' '.join(verify_cmd))


This section is the repo-level parity snapshot for the maintained full sample.

Read it in this order:
- the first table tells you which audit artifact and run root were used,
- the second table shows audit-derived stage timings for the full sample,
- the verify summary and first failed paths show the current parity state against `inputs_and_outputs/InSAR_dataset_test`.

If you need to regenerate the audit instead of consuming the latest maintained artifact, use notebook `03`.


## Notebook guide

Use the notebook set as a progression starting here, not as duplicates:

- `00_pystamps_beginner_walkthrough.ipynb`: gentle introduction for new users.
- `01_pystamps_dataset_inspection.ipynb`: dataset structure and stage-1 inputs.
- `02_pystamps_stage_execution.ipynb`: stage-by-stage teaching view.
- `03_pystamps_verification.ipynb`: verification and audit workflows.
- `04_rust_end2end_parity_validation.ipynb`: native-backed parity and backend evidence.

This notebook is the map of the set. The others are the deep dives.

## Validation and parity

There are two different levels of correctness work in this repo:

- Fresh-clone validation: `pytest`, build, and distribution checks.
- Full parity maintenance: audit and verify flows against local maintained datasets.

Important constraints:
- Full parity work depends on optional local datasets under `inputs_and_outputs/`.
- Some later-stage or audit flows also depend on external tools such as `triangle` and `snaphu`.
- `verify` is lighter and compares one run folder against one golden folder.
- `audit` is stricter and records fresh evidence artifacts under `inputs_and_outputs/validation_runs/`.

## How to use the library from Python

Most users should stay on the CLI, but the internal API is useful for notebooks, tests, and controlled orchestration.

Publicly useful pieces include:
- `RunConfig` and `load_config` for runtime configuration.
- `PipelineContext` and `run_pipeline` for orchestrated execution.
- `verify_run_against_golden` for explicit comparison.
- `pystamps.kernels.*` for direct kernel inspection or controlled experiments.

In [ ]:
if False:
    from pathlib import Path
    from pystamps.config import RunConfig
    from pystamps.pipeline.types import PipelineContext
    from pystamps.pipeline.stages import run_pipeline
    from pystamps.verify import verify_run_against_golden

    dataset = Path('inputs_and_outputs/InSAR_dataset_test').resolve()
    config = RunConfig()
    context = PipelineContext(
        dataset_root=dataset,
        run_config=config,
        start_step=1,
        end_step=2,
        dry_run=True,
    )
    report = run_pipeline(context)
    verify_report = verify_run_against_golden(dataset, dataset, config.tolerance)
    print(report.results)
    print(verify_report.ok)


## What to open next

- If you are new, open `00_pystamps_beginner_walkthrough.ipynb`.
- If you have a dataset in hand, run `pystamps status` and then open notebook `01` or `02`.
- If you care about output correctness, open notebook `03`.
- If you care about acceleration and native parity, open notebook `04`.

This notebook is the shortest path to knowing where to look next.